# AnyMatch — Inference on real AllianceChicago candidate pairs

Joins the blocking output (`candidate_pairs_*.parquet`, PATID_A / PATID_B only) with the cleaned MDM population (`MDM_Population_cleaned_v1.csv`), filters to pairs where **both** records have `valid_record = True`, optionally subsamples, and scores the result with the trained GPT-2 checkpoint via `predict_alliance.py`.

**Where to run**: locally, from the `AnyMatch/` folder. Launch Jupyter / VS Code with cwd = the `AnyMatch/` directory so relative paths resolve.

**Hardware**: CPU is fine for hundreds of pairs (~700 ms/pair from the synthetic-set benchmark). At that rate, the full 212k candidate set is ~40 hours locally — move to Colab Pro GPU for the full run.

**PHI reminder**: this notebook reads real patient data. Run only on a machine that satisfies your institution's policy.

## 1. Config

The only cell you normally edit. `N_PAIRS = -1` to score every valid pair; any positive int subsamples the first N (deterministic). `FEATURE_COLS` are positional — `_l` and `_r` must align column-for-column, which is automatic since both sides are joined from the same source.

In [ ]:
import os

# === EDIT THESE IF NEEDED ===
CKPT_DIR     = 'saved_models/anymatch_all9_gpt2_mode4'
PAIRS_PATH   = 'data/alliance/candidate_pairs_v1_20260521_035346.parquet'
RECORDS_PATH = 'data/alliance/MDM_Population_cleaned_v1.csv'
OUTPUT_DIR   = 'data/alliance'
OUTPUT_STEM  = 'anymatch_predictions'   # final name: {OUTPUT_STEM}_v1_{YYYYMMDD_HHMMSS}.csv

N_PAIRS = 100   # set to -1 to score every valid pair

# Source-to-prompt column rename. mode4 emits "<attr_name>: <value>" for each
# attribute, where <attr_name> comes from the dataframe column name (sans _l/_r).
# The training data uses clean lowercase English (name, price, manufacturer, ...)
# so we map our technical MDM column names to the same style to avoid train/serve
# mismatch. The dict order also defines the positional column order downstream.
#
# Name field uses full_name_tokens (a sorted set of name tokens from First/Middle/Last)
# rather than full_name_compact. Tokens is order- and bucket-invariant, which handles
# Hispanic two-surname shuffling (paternal vs. maternal landing in middle vs. last)
# and name-order swaps without per-field disagreement noise. Middle name is not sent
# as its own attribute because in this FQHC population middle is the noisiest bucket
# (where the maternal surname inconsistently lands) — its content rides through
# inside name_tokens.
FEATURE_RENAMES = {
    'full_name_tokens':    'name',          # sorted, space-joined set of name tokens
    'BirthDT_clean':       'dob',
    'SexAtBirthDSC_clean': 'sex',
    'SSN_clean':           'ssn',           # full 9-digit; NaN for the ~80% where it's missing
    'last_4_SSN':          'ssn last 4',    # backup signal for the 80% with no full SSN
    'AddressLine1_clean':  'address',
    'CityNM_clean':        'city',
    'StateCD_clean':       'state',
    'ZipCD_clean_base':    'zip',
    'Phones_set':          'phone',         # serialized to whitespace-joined string in cell 8
    'Email_clean':         'email',
}
FEATURE_COLS_SRC = list(FEATURE_RENAMES.keys())     # original MDM column names
FEATURE_COLS     = list(FEATURE_RENAMES.values())   # semantic names — drive everything downstream
# ============================

assert os.path.exists('loo.py'), (
    f'Expected to run from the AnyMatch/ directory (cwd={os.getcwd()!r}). '
    "cd into AnyMatch/ before launching Jupyter, or os.chdir() here."
)

needed = ['config.json', 'vocab.json', 'merges.txt']
missing = [f for f in needed if not os.path.exists(os.path.join(CKPT_DIR, f))]
if missing:
    raise FileNotFoundError(
        f'Missing from {CKPT_DIR}: {missing}. '
        'Download the trained checkpoint folder and place it there.'
    )
for p in [PAIRS_PATH, RECORDS_PATH]:
    assert os.path.exists(p), f'Missing input: {p}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config OK.')

## 2. Load records, filter to `valid_record = True`

Indexing by PATID makes the join in cell 4 a single `.loc[...]` lookup per side.

In [ ]:
import pandas as pd

# Force-read numeric-looking ID columns as strings. Without this, pandas infers
# them as float64 (because of NaN in the column), which prints values as
# "358467965.0" and silently drops leading zeros — wrecks exact-match signal.
ID_STR_COLS = {
    'PATID':             'string',
    'SSN_clean':         'string',
    'last_4_SSN':        'string',
    'ZipCD_clean_base':  'string',
    'ZipCD_clean_ext':   'string',
    'PrimaryPhoneNBR_clean': 'string',
    'Phone01NBR_clean':  'string',
    'Phone02NBR_clean':  'string',
    'Phone03NBR_clean':  'string',
}

records = pd.read_csv(RECORDS_PATH, low_memory=False, dtype=ID_STR_COLS)
records_valid = records[records['valid_record']].copy()
print(f'{len(records):,} total records → {len(records_valid):,} with valid_record=True')

missing_cols = [c for c in FEATURE_COLS_SRC if c not in records_valid.columns]
if missing_cols:
    raise KeyError(f'FEATURE_COLS_SRC not found in records: {missing_cols}')

# Apply the source-to-semantic rename so every downstream consumer
# (per-record transforms, join, intermediate CSV, df_serializer, output display)
# sees the prompt-ready names.
records_valid = records_valid.rename(columns=FEATURE_RENAMES)

records_valid = records_valid.set_index('PATID')
if not records_valid.index.is_unique:
    dup = records_valid.index[records_valid.index.duplicated()].unique()[:5]
    raise ValueError(f'PATID is not unique in MDM file. Examples: {list(dup)}')

# Sanity: ID columns should be strings now (or NaN), no trailing ".0".
for col in ['ssn', 'ssn last 4', 'zip']:
    if col in records_valid.columns:
        sample = records_valid[col].dropna().head(3).tolist()
        print(f'  {col}: dtype={records_valid[col].dtype}, sample={sample}')

## 3. Load pairs, keep only pairs where both sides are valid, subsample

In [ ]:
pairs = pd.read_parquet(PAIRS_PATH)[['PATID_A', 'PATID_B']]
pairs = pairs.sample(frac=1, random_state=42)
print(f'{len(pairs):,} candidate pairs from blocking')

valid_ids = set(records_valid.index)
pairs = pairs[pairs['PATID_A'].isin(valid_ids) & pairs['PATID_B'].isin(valid_ids)].reset_index(drop=True)
print(f'{len(pairs):,} pairs where both sides have valid_record=True')

if N_PAIRS != -1:
    pairs = pairs.head(N_PAIRS)
    print(f'Subsampled to first {len(pairs):,} pairs (N_PAIRS={N_PAIRS})')

## 4. Build the `_l` / `_r` wide CSV that `predict_alliance.py` expects

Two source columns are stored as Python `set` literals on disk and need to be flattened to whitespace-joined strings before serialization:

- `full_name_tokens` → sorted, space-joined string. Sorting gives a canonical, order-invariant representation so two records with the same name tokens in different bucket assignments (Hispanic surname shuffle, name-order swap) produce identical prompts.
- `Phones_set` → sorted, space-joined string of plain digits. Without this, pandas writes the literal `{'3125551234'}` to CSV and the braces / quotes pollute the prompt.

In [ ]:
import ast

def serialize_set_field(x):
    """Flatten a Python set-on-disk into a sorted, space-joined string.
    Handles three input shapes: a live set/list/tuple, a stringified literal
    like "{'ANNE', 'MARIE'}" (what pandas writes when a set goes through CSV),
    and plain strings / NaN. Sorting ensures the output is canonical so two
    records with the same tokens in different orders / buckets serialize
    identically."""
    if isinstance(x, (set, list, tuple)):
        return ' '.join(sorted(str(t) for t in x if t and str(t).lower() != 'nan'))
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if not s or s.lower() == 'nan':
        return ''
    if s.startswith(('{', '[', '(')):
        try:
            parsed = ast.literal_eval(s)
            return ' '.join(sorted(str(t) for t in parsed if t and str(t).lower() != 'nan'))
        except (ValueError, SyntaxError):
            return s
    return s

# Per-record field transforms — done ONCE on records_valid, not per-side.
# Doing them per-side previously caused dob_r to come out empty because
# pd.to_datetime(series).dt.strftime(...) preserves the input series' index, and
# assigning that back into side_df triggered index-alignment with the side_df
# range index, dropping all values for the right side. Moving the transform up
# eliminates the alignment surprise and saves N-fold work.
records_valid = records_valid.copy()
records_valid['name'] = records_valid['name'].apply(serialize_set_field)
records_valid['phone'] = records_valid['phone'].apply(serialize_set_field)
records_valid['dob'] = (
    pd.to_datetime(records_valid['dob'], errors='coerce').dt.strftime('%Y-%m-%d')
)

wide = pairs.reset_index(drop=True).copy()
for side, patid_col in [('l', 'PATID_A'), ('r', 'PATID_B')]:
    side_df = (
        records_valid.loc[pairs[patid_col].values, FEATURE_COLS]
        .reset_index(drop=True)
        .add_suffix(f'_{side}')
    )
    wide = pd.concat([wide, side_df], axis=1)

# Sanity check — both DOB and name columns should be populated where the source was.
for col in ['dob_l', 'dob_r', 'name_l', 'name_r']:
    n_pop = wide[col].astype('string').str.len().gt(0).sum()
    print(f'{col}: {n_pop}/{len(wide)} populated')

intermediate_csv = f'{OUTPUT_DIR}/_anymatch_input_temp.csv'
wide.to_csv(intermediate_csv, index=False)
print(f'Wrote {len(wide):,} rows × {len(wide.columns)} cols to {intermediate_csv}')
wide.head(3)

## 5. Peek at the model input

Before running inference, build the exact prompt strings that AnyMatch will see for the first few pairs. Useful for sanity-checking that columns line up, missing values are showing as `N/A` (not literal `nan` strings), and the token count stays under GPT-2's 1024-token cap.

In [ ]:
# Build the exact prompt that AnyMatch will see for each pair.
# df_serializer (mode4) emits one string per row using the template:
#   "Given the attributes of two records, are they the same?
#    Record A is name: <v>, dob: <v>, ssn: <v>, ....
#    Record B is name: <v>, dob: <v>, ssn: <v>, ...."
# The attribute names ("name", "dob", "ssn", ...) come from the dataframe column
# names (sans _l/_r) — which is why we renamed in cell 4. Missing values become
# the literal token "N/A" via .fillna('N/A').
# PATID columns are bystanders — they ride through the CSV but are NOT in the prompt.
from utils.data_utils import df_serializer

preview = wide.copy()
if 'label' not in preview.columns:
    preview['label'] = 0   # df_serializer requires the column; placeholder only

serialized = df_serializer(preview, 'mode4')
N_PREVIEW = min(3, len(serialized))
for i in range(N_PREVIEW):
    print(f'=== Pair {i}  PATID_A={wide["PATID_A"].iloc[i]}  PATID_B={wide["PATID_B"].iloc[i]} ===')
    print(serialized['text'].iloc[i])
    print()

# Quick token-budget check (GPT-2 hard limit is 1024 tokens per sequence).
# from transformers import GPT2Tokenizer
# _tok = GPT2Tokenizer.from_pretrained(CKPT_DIR)
# _lens = serialized['text'].apply(lambda t: len(_tok.encode(t)))
# print(f'Token lengths — min={_lens.min()}, mean={_lens.mean():.0f}, '
#       f'max={_lens.max()} (GPT-2 cap = 1024)')

## 6. Run inference

Uses the existing `predict_alliance.py` CLI: loads the GPT-2 checkpoint, serializes each row with `df_serializer` (`mode4`), and appends `pred` (0/1) and `match_prob` (P(label=1)) columns. PATID_A / PATID_B ride through untouched.

In [ ]:
import sys, time
from datetime import datetime

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
output_csv = f'{OUTPUT_DIR}/{OUTPUT_STEM}_v1_{ts}.csv'

t0 = time.perf_counter()
!{sys.executable} predict_alliance.py \
    --model_path {CKPT_DIR} \
    --base_model gpt2 \
    --serialization_mode mode4 \
    --input_csv {intermediate_csv} \
    --output_csv {output_csv} \
    --batch_size 32
elapsed = time.perf_counter() - t0

print(f'\n{len(wide):,} pairs in {elapsed:.1f}s ({elapsed/max(len(wide),1)*1000:.0f} ms/pair)')
print(f'Output: {output_csv}')

## 7. Inspect predictions

No ground-truth labels here, so no F1. Useful diagnostics:
- pred distribution
- `match_prob` histogram
- top-10 most confident matches (sanity check that obvious matches score high)
- 10 pairs closest to 0.5 (threshold-calibration candidates / hand-label targets)

In [ ]:
out = pd.read_csv(output_csv)
print('--- pred distribution ---')
print(out['pred'].value_counts())
print('\n--- match_prob summary ---')
display(out['match_prob'].describe())
out['match_prob'].plot.hist(bins=50, title='match_prob distribution');

In [ ]:
alt_cols = [c for col in FEATURE_COLS for c in (f'{col}_l', f'{col}_r')]
view_cols = ['match_prob', 'PATID_A', 'PATID_B', *alt_cols]

with pd.option_context('display.max_columns', None, 'display.width', None):
    print('--- Top-10 most confident matches ---')
    display(out.nlargest(10, 'match_prob')[view_cols].round({'match_prob': 3}))

    print('--- 10 pairs closest to 0.5 (uncertain) ---')
    near_idx = (out['match_prob'] - 0.5).abs().sort_values().index[:10]
    display(out.loc[near_idx, view_cols].round({'match_prob': 3}))

## 8. Notes

- **Threshold tuning**: `pred` uses 0.5 by default. For production, sweep `match_prob` against a hand-labeled holdout to hit your precision / recall target.
- **Scaling**: at the synthetic-set rate (~700 ms/pair on Mac CPU), the full 212k candidate set is ~40 hours locally. For the full run, copy `_anymatch_input_temp.csv` to Drive and use `anymatch_colab.ipynb` Section 5 on a Colab Pro GPU.
- **Schema invariant**: column order in `FEATURE_COLS` is positional. If you add or reorder columns, both sides re-emit symmetrically — no separate L/R definition to keep in sync.
- The temp file `_anymatch_input_temp.csv` is left on disk for debugging; safe to delete after the run.